In [1]:
import pandas as pd

In [2]:
CLEANED_PATH = r"C:\glassIDS\data\processed\cicids2017_cleaned.csv"

df = pd.read_csv(CLEANED_PATH)

print(df.shape)
print(df.columns.tolist())
print(df["Label"].value_counts())

(2520798, 81)
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', '

In [3]:
diag = (
    df.groupby("Label")
      .apply(lambda g: pd.Series({"min_idx": g.index.min(), "max_idx": g.index.max(), "count": len(g)}))
      .sort_values("min_idx")
)
print(diag)

                            min_idx  max_idx    count
Label                                                
BENIGN                            0  2520797  2095057
DDoS                          18588   195651   128014
PortScan                     224465   428475    90694
Bot                          458131   615422     1948
Infiltration                1179734  1304977       36
Web Attack - Brute Force    1375197  1421605     1470
Web Attack - XSS            1430597  1443644      652
Web Attack - Sql Injection  1447444  1450208       21
FTP-Patator                 1531378  1673321     5931
SSH-Patator                 1673323  1930239     3219
DoS slowloris               1936609  1994955     5385
DoS Slowhttptest            1994957  2000263     5228
DoS Hulk                    2000264  2196863   172846
DoS GoldenEye               2196864  2520791    10286
Heartbleed                  2438544  2439142       11


In [7]:
from sklearn.model_selection import train_test_split

feature_cols = []
for c in df.columns:
    if c not in ("Label", "is_attack", "source_file", "Destination Port"):
        feature_cols.append(c)
        
X = df[feature_cols]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42  #stratify for proper distribution of attacks in train test split
)

print(y_train.value_counts())
print(y_test.value_counts())

Label
BENIGN                        1676045
DoS Hulk                       138277
DDoS                           102411
PortScan                        72555
DoS GoldenEye                    8229
FTP-Patator                      4745
DoS slowloris                    4308
DoS Slowhttptest                 4182
SSH-Patator                      2575
Bot                              1558
Web Attack - Brute Force         1176
Web Attack - XSS                  522
Infiltration                       29
Web Attack - Sql Injection         17
Heartbleed                          9
Name: count, dtype: int64
Label
BENIGN                        419012
DoS Hulk                       34569
DDoS                           25603
PortScan                       18139
DoS GoldenEye                   2057
FTP-Patator                     1186
DoS slowloris                   1077
DoS Slowhttptest                1046
SSH-Patator                      644
Bot                              390
Web Attack - Brute For

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

for i, cls in enumerate(le.classes_):
    print(i, cls)

0 BENIGN
1 Bot
2 DDoS
3 DoS GoldenEye
4 DoS Hulk
5 DoS Slowhttptest
6 DoS slowloris
7 FTP-Patator
8 Heartbleed
9 Infiltration
10 PortScan
11 SSH-Patator
12 Web Attack - Brute Force
13 Web Attack - Sql Injection
14 Web Attack - XSS


In [14]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train_enc)

In [15]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
)

model.fit(X_train, y_train_enc, sample_weight=sample_weights)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklear

In [16]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, recall_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# Per-class precision/recall/F1 — this table IS the Week 1 results table
print(classification_report(y_test_enc, y_pred, target_names=le.classes_, digits=3))

# Macro ROC-AUC (one-vs-rest, unweighted across classes)
macro_auc = roc_auc_score(y_test_enc, y_proba, multi_class="ovr", average="macro")
print(f"Macro ROC-AUC (OvR): {macro_auc:.4f}")

# Full confusion matrix, labeled
cm = confusion_matrix(y_test_enc, y_pred)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print(cm_df)

                            precision    recall  f1-score   support

                    BENIGN      1.000     0.998     0.999    419012
                       Bot      0.494     0.992     0.659       390
                      DDoS      0.999     1.000     1.000     25603
             DoS GoldenEye      0.987     0.999     0.993      2057
                  DoS Hulk      0.998     0.999     0.999     34569
          DoS Slowhttptest      0.932     0.995     0.963      1046
             DoS slowloris      0.988     0.991     0.989      1077
               FTP-Patator      0.998     0.999     0.999      1186
                Heartbleed      1.000     1.000     1.000         2
              Infiltration      0.875     1.000     0.933         7
                  PortScan      0.989     1.000     0.994     18139
               SSH-Patator      0.998     1.000     0.999       644
  Web Attack - Brute Force      0.767     0.816     0.791       294
Web Attack - Sql Injection      0.429     0.750

In [17]:
y_test_binary = (y_test != "BENIGN").astype(int)
y_pred_binary = (le.inverse_transform(y_pred) != "BENIGN").astype(int)

attack_recall = recall_score(y_test_binary, y_pred_binary)
print(f"Attack-detection recall (any attack type vs BENIGN): {attack_recall:.4f}")

Attack-detection recall (any attack type vs BENIGN): 0.9998
